# nb05_submit : Late Submission (인터넷 OFF) — 단일 실험 · 다중 모델 · P1 앙상블

- **가속기 : GPU (T4)** · **인터넷 : OFF (필수, 대회 규칙 : ON으로 저장된 버전은 Submit 불가)**
- **입력** : 대회 데이터, Dataset `hms-src`, 모델 Output 1개
  - 단일 실험 (`PRESET="E4"` 등) : 노트북 `Nb04_train` Output (해당 실험 버전)
  - 다중 모델·앙상블 (`PRESET="E4E5"`, `"E8"`) : 노트북 `nb08_pack` Output (models/E4, E5, P1 + e8_recipe.json)
- **입력 방식 자동 인식** : 실험별 metrics.json의 settings.input → spec(10분) / eeg(50초) / both
- **출력** : `/kaggle/working/submission.csv`

| 섹션 | 내용 |
|---|---|
| 0 | 설정 : PRESET → 레시피(모델 목록, 딥러닝 비중, P1 사용, T), 입력 탐색 |
| 1 | 테스트 전처리 : 학습과 같은 함수(eeg_segment, spec 변환), P1용 프로파일 지표 |
| 2 | 예측 : 실험별 fold 평균 → 실험 평균 → (P1) w·DL + (1−w)·P1 → T |
| 3 | 일관성 점검 : 학습 데이터 32건을 같은 경로로 다시 예측해 OOF와 비교 (딥러닝 실험별 + P1) |
| 4 | submission.csv 저장·검증 |

## 0. 설정 (목적 : 제출 조합 지정·입력 탐색)

In [ ]:
PRESET = "E8"          # "E2"~"E6" : 단일 실험 / "E4E5" : E4·E5 평균 (+T) / "E8" : e8_recipe.json (E4·E5 + P1v2)

PRESETS = {
    "E4E5": {"models": ["E4", "E5"], "dl_weight": 1.0, "use_p1": False, "temperature": 1.09},   # nb07 v2 : E4+E5+T, T fold별 1.08~1.10
}
for e in ["E1", "E2", "E3", "E4", "E5", "E6"]:
    PRESETS[e] = {"models": [e], "dl_weight": 1.0, "use_p1": False, "temperature": 1.0}

import sys, os, time, json
from pathlib import Path

def find_first(pattern_tail, root="/kaggle/input", depth=6):
    for d in range(1, depth + 1):
        hits = sorted(Path(root).glob("/".join(["*"] * d) + "/" + pattern_tail))
        if hits:
            return hits[0]
    return None

src_hit = find_first("src/data/targets.py")
assert src_hit, "hms-src를 찾지 못했습니다. Add Input에서 hms-src를 연결하세요."
SRC_ROOT = src_hit.parents[2]
sys.path.insert(0, str(SRC_ROOT))

if PRESET == "E8":
    recipe_hit = find_first("e8_recipe.json")
    assert recipe_hit, "e8_recipe.json을 찾지 못했습니다. Add Input에서 nb08_pack Output을 연결하세요."
    RECIPE = json.load(open(recipe_hit))
else:
    RECIPE = dict(PRESETS[PRESET])

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
from src.data.load import find_data_dir, spec_window
from src.data.build import eeg_segment
from src.data.targets import build_targets, select_representative, PROB_COLS, CLASSES
from src.models.eeg_spec import HMSNet

DATA_DIR = find_data_dir()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXP_DIRS, EXP_INPUT, EXP_NOUT = {}, {}, {}
for exp in RECIPE["models"]:
    hit = find_first(f"models/{exp}/fold0.pt")
    assert hit, f"models/{exp}/fold0.pt를 찾지 못했습니다. 모델 Output(Nb04_train 또는 nb08_pack) 연결을 확인하세요."
    EXP_DIRS[exp] = hit.parent
    with open(hit.parent / "metrics.json") as f:
        st = json.load(f)["settings"]
    EXP_INPUT[exp] = st.get("input", "spec")                            # E1·E2 metrics.json에는 input 항목 없음 → spec
    EXP_NOUT[exp] = len(CLASSES) + (2 if st.get("aux") else 0)          # E6 : 보조 헤드 2개 포함 (예측에는 앞 6개만 사용)
P1_DIR = None
if RECIPE["use_p1"]:
    hit = find_first("models/P1/p1_meta.json")
    assert hit, "models/P1/p1_meta.json을 찾지 못했습니다. nb08_pack Output을 연결하세요."
    P1_DIR = hit.parent
    P1_META = json.load(open(hit))

NEED_SPEC = RECIPE["use_p1"] or any(v in ("spec", "both") for v in EXP_INPUT.values())
NEED_EEG = RECIPE["use_p1"] or any(v in ("eeg", "both") for v in EXP_INPUT.values())
try:                                                       # 인터넷 ON 저장 버전은 제출 불가 → 미리 경고
    import socket; socket.create_connection(("www.kaggle.com", 443), timeout=3).close()
    print("[경고] 인터넷이 켜져 있습니다. Settings → Internet Off 후 Save Version 해야 Submit 가능합니다.")
except OSError:
    pass
print(f"PRESET={PRESET} | 레시피 {RECIPE}")
print(f"DEVICE={DEVICE} | DATA_DIR={DATA_DIR}\nSRC_ROOT={SRC_ROOT}")
for exp, d in EXP_DIRS.items():
    print(f"  {exp} : {d} | fold 가중치 {len(list(d.glob('fold[0-9].pt')))}개 | 입력 {EXP_INPUT[exp]}")
if P1_DIR is not None:
    print(f"  P1 : {P1_DIR} | fold 모델 {len(list(P1_DIR.glob('fold[0-9].txt')))}개 | 입력 {len(P1_META['x_cols'])}열")

## 1. 전처리 (목적 : 학습(nb02·nb03·nb04)과 동일한 입력 생성)

In [ ]:
def spec10(spec_df, offset):
    """10분 스펙트로그램 → nb02 저장 형식과 같은 (4, 100, 300) float16 log1p 배열 (P1 지표 입력)."""
    return np.log1p(np.clip(spec_window(spec_df, offset), 0, None)).astype(np.float16)

def spec_image(s10):
    """nb04 입력 : float32 변환 후 4부위 세로 연결 (1, 400, 300), 표본별 표준화."""
    x = s10.astype(np.float32).reshape(-1, 300)
    return ((x - x.mean()) / (x.std() + 1e-6))[None]

def build_inputs(df, split, spec_offsets=None, eeg_offsets=None):
    """행마다 (스펙트로그램 이미지, 50초 몽타주, P1 입력 행). 테스트 파일은 이미 10분·50초 구간 (offset 0)."""
    XS, XE, XP = [], [], []
    for i, r in enumerate(df.itertuples()):
        s = e = qc = None
        if NEED_SPEC:
            spec = pd.read_parquet(DATA_DIR / f"{split}_spectrograms" / f"{r.spectrogram_id}.parquet")
            s = spec10(spec, float(spec["time"].min()) if spec_offsets is None else spec_offsets[i])
            XS.append(spec_image(s))
        if NEED_EEG:
            eeg = pd.read_parquet(DATA_DIR / f"{split}_eegs" / f"{r.eeg_id}.parquet")
            e, qc = eeg_segment(eeg, 0.0 if eeg_offsets is None else eeg_offsets[i])   # float16 (16, 10000) : nb02와 같음
            XE.append(e.astype(np.float32))
        if P1_DIR is not None:
            f = profile_features(e, s)                                                  # nb03과 같은 float16 입력
            f.update({k: qc[k] for k in P1_META["qc"]})
            XP.append([f[c] for c in P1_META["x_cols"]])
    stack = lambda a: np.stack(a).astype(np.float32) if a else None
    return stack(XS), stack(XE), (np.array(XP, dtype=np.float64) if XP else None)

if P1_DIR is not None:
    from src.profile.qeeg import profile_features
    import lightgbm as lgb

t0 = time.time()
test = pd.read_csv(DATA_DIR / "test.csv")
sub_template = pd.read_csv(DATA_DIR / "sample_submission.csv")
XS_test, XE_test, XP_test = build_inputs(test, "test")
for name, a in [("spec", XS_test), ("eeg", XE_test), ("P1", XP_test)]:
    if a is not None:
        print(f"테스트 {len(test)}건 | {name} 입력 {a.shape} | 평균 {a.mean():.3f}")
print(f"전처리 {time.time() - t0:.1f}s")

## 2. 예측 (목적 : 실험별 fold 평균 → 실험 평균 → P1 결합 → 온도 보정)

In [ ]:
def load_model(path, input_mode, n_out):
    bb = timm.create_model("efficientnet_b0", pretrained=False, in_chans=1, num_classes=n_out)  # 인터넷 불필요
    bb.load_state_dict(torch.load(path, map_location="cpu"))       # 저장 가중치는 백본만 (E1~E5 공통 형식)
    return HMSNet(bb, input_mode).to(DEVICE).eval()

@torch.no_grad()
def predict(model, XS, XE, bs=64):
    n, out = len(XS if XS is not None else XE), []
    for i in range(0, n, bs):
        s = torch.from_numpy(XS[i:i + bs]).to(DEVICE) if model.input_mode != "eeg" else None
        e = torch.from_numpy(XE[i:i + bs]).to(DEVICE) if model.input_mode != "spec" else None
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out.append(F.softmax(model(s, e).float()[:, :len(CLASSES)], dim=1).cpu().numpy())   # 보조 헤드 출력은 제외
    return np.concatenate(out)

def temp(p, T):
    if T == 1.0:
        return p
    z = np.log(np.clip(p, 1e-7, 1)) / T
    z = np.exp(z - z.max(axis=1, keepdims=True))
    return z / z.sum(axis=1, keepdims=True)

t0 = time.time()
MODELS = {exp: {int(p.stem[4:]): load_model(p, EXP_INPUT[exp], EXP_NOUT[exp]) for p in sorted(d.glob("fold[0-9].pt"))}
          for exp, d in EXP_DIRS.items()}                            # fold0~4.pt만 (stage1 가중치 제외)
pred_exp = {exp: np.mean([predict(m, XS_test, XE_test) for m in ms.values()], axis=0) for exp, ms in MODELS.items()}
pred_dl = np.mean(list(pred_exp.values()), axis=0)                   # 실험 같은 비중 평균
pred = pred_dl
if P1_DIR is not None:
    P1_MODELS = {int(p.stem[4:]): lgb.Booster(model_file=str(p)) for p in sorted(P1_DIR.glob("fold[0-9].txt"))}
    pred_p1 = np.mean([b.predict(XP_test) for b in P1_MODELS.values()], axis=0)
    w = RECIPE["dl_weight"]
    pred = w * pred_dl + (1 - w) * pred_p1
pred_test = temp(pred, RECIPE["temperature"])
print(f"예측 완료 {time.time() - t0:.1f}s | 실험 {list(MODELS)} (fold {[len(v) for v in MODELS.values()]})"
      + (f" + P1 fold {len(P1_MODELS)}개, 딥러닝 비중 {RECIPE['dl_weight']}" if P1_DIR is not None else "")
      + f" | T {RECIPE['temperature']} | 예측 합 {pred_test.sum(axis=1).round(4)[:3]}")

## 3. 일관성 점검 (목적 : 학습·추론 전처리 불일치 탐지, 공개 테스트 1건일 때만 실행)

In [ ]:
# 학습 데이터 32건을 같은 추론 경로로 다시 예측해 학습 때 저장한 OOF와 비교 (각 행은 자신이 검증 fold였던 모델로 예측).
# 딥러닝 실험별 + P1(테스트용 지표 산출 경로까지 점검). 차이가 크면 전처리 불일치 → 제출 중단. 숨김 테스트 재실행 때는 건너뜀.
if len(test) < 10:
    rep = select_representative(build_targets(pd.read_csv(DATA_DIR / "train.csv")))
    cols = ["eeg_id", "spectrogram_id", "spectrogram_label_offset_seconds", "eeg_label_offset_seconds"]
    first = pd.read_csv(EXP_DIRS[RECIPE["models"][0]] / "oof.csv")
    chk = first.sample(min(32, len(first)), random_state=0)[["eeg_id", "fold"]].merge(rep[cols], on="eeg_id")
    XS, XE, XP = build_inputs(chk, "train", chk.spectrogram_label_offset_seconds.to_numpy(), chk.eeg_label_offset_seconds.to_numpy())
    PRED = [f"pred_{c}" for c in CLASSES]
    checks = {}
    for exp, ms in MODELS.items():
        ref = pd.read_csv(EXP_DIRS[exp] / "oof.csv").set_index("eeg_id").loc[chk.eeg_id, PRED].to_numpy()
        p = np.zeros_like(ref)
        for k in chk.fold.unique():
            idx = (chk.fold == k).to_numpy()
            p[idx] = predict(ms[int(k)], XS[idx] if XS is not None else None, XE[idx] if XE is not None else None)
        checks[exp] = np.abs(p - ref).max()
    if P1_DIR is not None:
        ref = pd.read_csv(P1_DIR / "oof.csv").set_index("eeg_id").loc[chk.eeg_id, PRED].to_numpy()
        p = np.zeros_like(ref)
        for k in chk.fold.unique():
            idx = (chk.fold == k).to_numpy()
            p[idx] = P1_MODELS[int(k)].predict(XP[idx])
        checks["P1"] = np.abs(p - ref).max()
    for name, d in checks.items():
        print(f"OOF 재현 점검 {name} : {len(chk)}건 | 최대 차이 {d:.4f}")
    assert max(checks.values()) < 0.02, "학습 때 예측과 다릅니다 : 전처리 불일치 가능성 → 제출 중단"
else:
    print("일관성 점검 건너뜀 (숨김 테스트 실행)")

## 4. 제출 파일 (목적 : 양식 검증 후 저장)

In [ ]:
sub = pd.DataFrame(pred_test, columns=[f"{c}_vote" for c in CLASSES])
sub.insert(0, "eeg_id", test.eeg_id.values)
sub = sub[sub_template.columns]                                     # 양식과 같은 열 순서

assert len(sub) == len(test) and not sub.isna().any().any()
assert np.allclose(sub.drop(columns="eeg_id").sum(axis=1), 1, atol=1e-4)
sub.to_csv("/kaggle/working/submission.csv", index=False)
display(sub.head())
print(f"submission.csv 저장 : {len(sub)}행 | PRESET={PRESET} | {RECIPE.get('candidate', '+'.join(RECIPE['models']))}")

## 다음 단계
1. 설정 확인 : **Internet OFF**, Accelerator **GPU**, 입력 3개(대회 데이터, hms-src, 모델 Output)
2. 셀 0의 `PRESET` 확인 → **Save Version → Save & Run All**
3. 완료된 **새 버전** Output의 submission.csv → **Submit** (한 번만)
4. My Submissions에서 **Public / Private 점수** 확인 → results.csv에 기록